In [1]:
import os
import zipfile
import re
import math

import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, f1_score

In [2]:
# File paths
train_text_path = "train_text.txt"
train_labels_path = "train_labels.txt"

val_text_path = "val_text.txt"
val_labels_path = "val_labels.txt"

test_text_path = "test_text.txt"
test_labels_path = "test_labels.txt"


# Load TweetEval text files
def load_tweets(filename):
    with open(filename, "r", encoding="utf-8") as file:
        return [line.rstrip("\n") for line in file]


train_tweets = load_tweets(train_text_path)
val_tweets = load_tweets(val_text_path)
test_tweets = load_tweets(test_text_path)

In [3]:
train_labels = pd.read_csv(
    train_labels_path,
    header=None,
    names=["label"]
)

val_labels = pd.read_csv(
    val_labels_path,
    header=None,
    names=["label"]
)

test_labels = pd.read_csv(
    test_labels_path,
    header=None,
    names=["label"]
)

In [4]:
print("Training tweets:", len(train_tweets))
print("Validation tweets:", len(val_tweets))
print("Test tweets:", len(test_tweets))

print("Training labels:", len(train_labels))
print("Validation labels:", len(val_labels))
print("Test labels:", len(test_labels))

Training tweets: 45615
Validation tweets: 2000
Test tweets: 12284
Training labels: 45615
Validation labels: 2000
Test labels: 12284


In [5]:
train_df = pd.DataFrame({
    "tweet": train_tweets,
    "label": train_labels["label"]
})

val_df = pd.DataFrame({
    "tweet": val_tweets,
    "label": val_labels["label"]
})

test_df = pd.DataFrame({
    "tweet": test_tweets,
    "label": test_labels["label"]
})

In [6]:
train_df.head()

,tweet,label
0,"""QT @user In the original draft of the 7th boo...",2
1,"""Ben Smith / Smith (concussion) remains out of...",1
2,Sorry bout the stream last night I crashed out...,1
3,Chase Headley's RBI double in the 8th inning o...,1
4,@user Alciato: Bee will invest 150 million in ...,2


In [7]:
print("Training label distribution:")
print(train_df["label"].value_counts().sort_index())

print("\nValidation label distribution:")
print(val_df["label"].value_counts().sort_index())

print("\nTest label distribution:")
print(test_df["label"].value_counts().sort_index())

Training label distribution:
label
0     7093
1    20673
2    17849
Name: count, dtype: int64

Validation label distribution:
label
0    312
1    869
2    819
Name: count, dtype: int64

Test label distribution:
label
0    3972
1    5937
2    2375
Name: count, dtype: int64


In [8]:
with open("mapping.txt", "r", encoding="utf-8") as file:
    print(file.read())

0	negative
1	neutral
2	positive



In [9]:
import zipfile
import pandas as pd

with zipfile.ZipFile("socialsent_hist_adj.zip", "r") as z:
    with z.open("adjectives/2000.tsv") as file:
        adj_2000 = pd.read_csv(file, sep="\t")

adj_2000.head()

,vile,-3.18,0.61
0,tragic,-3.15,0.66
1,ugly,-3.10,0.49
2,traumatic,-3.08,0.68
3,stinging,-3.05,0.56
4,awful,-2.89,0.49


In [10]:
print(adj_2000.columns)
print(adj_2000.shape)

Index(['vile', '-3.18', '0.61'], dtype='str')
(2041, 3)


In [11]:
with zipfile.ZipFile("socialsent_hist_freq.zip", "r") as z:
    with z.open("frequent_words/2000.tsv") as file:
        freq_2000 = pd.read_csv(file, sep="\t")

freq_2000.head()


,ugly,-3.90,1.16
0,painful,-3.69,1.53
1,intent,-3.49,1.67
2,terrible,-3.38,1.55
3,drunk,-3.28,1.16
4,tragedy,-3.26,1.48


In [12]:
print(freq_2000.columns)
print(freq_2000.shape)

Index(['ugly', '-3.90', '1.16'], dtype='str')
(4923, 3)


In [13]:
with zipfile.ZipFile("socialsent_subreddits.zip", "r") as z:
    with z.open("subreddits/3DS.tsv") as file:
        subreddit_3ds = pd.read_csv(file, sep="\t")

subreddit_3ds.head()

,ui,-6.06,1.11
0,shitty,-6.04,1.41
1,pathetic,-5.80,1.50
2,suck,-5.75,1.49
3,stupid,-5.72,1.37
4,lazy,-5.64,1.34


In [14]:
print(subreddit_3ds.columns)
print(subreddit_3ds.shape)

Index(['ui', '-6.06', '1.11'], dtype='str')
(4338, 3)


In [15]:
def load_lexicon_from_zip(zip_path, file_path):
    with zipfile.ZipFile(zip_path, "r") as z:
        with z.open(file_path) as file:
            return pd.read_csv(
                file,
                sep="\t",
                header=None,
                names=["word", "score", "std"]
            )


adj_2000 = load_lexicon_from_zip(
    "socialsent_hist_adj.zip",
    "adjectives/2000.tsv"
)

freq_2000 = load_lexicon_from_zip(
    "socialsent_hist_freq.zip",
    "frequent_words/2000.tsv"
)

subreddit_3ds = load_lexicon_from_zip(
    "socialsent_subreddits.zip",
    "subreddits/3DS.tsv"
)

In [16]:
print(adj_2000.head())
print()
print(freq_2000.head())
print()
print(subreddit_3ds.head())

        word  score   std
0       vile  -3.18  0.61
1     tragic  -3.15  0.66
2       ugly  -3.10  0.49
3  traumatic  -3.08  0.68
4   stinging  -3.05  0.56

       word  score   std
0      ugly  -3.90  1.16
1   painful  -3.69  1.53
2    intent  -3.49  1.67
3  terrible  -3.38  1.55
4     drunk  -3.28  1.16

       word  score   std
0        ui  -6.06  1.11
1    shitty  -6.04  1.41
2  pathetic  -5.80  1.50
3      suck  -5.75  1.49
4    stupid  -5.72  1.37


In [17]:
print("Historical adjectives:")
print(adj_2000["score"].describe())

print("\nHistorical frequent words:")
print(freq_2000["score"].describe())

print("\n3DS:")
print(subreddit_3ds["score"].describe())

Historical adjectives:
count    2041.000000
mean        0.000392
std         0.899240
min        -3.180000
25%        -0.410000
50%         0.020000
75%         0.460000
max         3.070000
Name: score, dtype: float64

Historical frequent words:
count    4924.000000
mean        0.000014
std         0.737869
min        -3.900000
25%        -0.290000
50%         0.050000
75%         0.350000
max         2.760000
Name: score, dtype: float64

3DS:
count    4339.000000
mean        0.000037
std         1.000155
min        -6.060000
25%        -0.360000
50%         0.150000
75%         0.555000
max         2.750000
Name: score, dtype: float64


In [18]:
# Files requested by the assignment
subreddit_files = [
    "3DS.tsv",
    "4chan.tsv",
    "2007scape.tsv",
    "ACTrade.tsv",
    "amiugly.tsv",
    "BabyBumps.tsv",
    "baseball.tsv",
    "canada.tsv",
    "CasualConversation.tsv",
    "DarknetMarkets.tsv",
    "darksouls.tsv",
    "elderscrollsonline.tsv",
    "Eve.tsv",
    "Fallout.tsv",
    "fantasyfootball.tsv",
    "GameDeals.tsv",
    "gamegrumps.tsv",
    "halo.tsv",
    "Homebrewing.tsv",
    "IAmA.tsv",
    "india.tsv",
    "jailbreak.tsv",
    "Jokes.tsv",
    "KerbalSpaceProgram.tsv",
    "Keto.tsv",
    "leagueoflegends.tsv",
    "Libertarian.tsv",
    "magicTCG.tsv",
    "MakeupAddiction.tsv",
    "Naruto.tsv",
    "nba.tsv",
    "occulus.tsv",
    "OkCupid.tsv",
    "Parenting.tsv",
    "pathofexile.tsv",
    "raisedbynarcissists.tsv",
    "Random_Acts_Of_Amazon.tsv",
    "science.tsv",
    "Seattle.tsv",
    "TalesFromRetail.tsv",
    "talesfromtechsupport.tsv",
    "ultrahardcore.tsv",
    "videos.tsv",
    "Warthunder.tsv",
    "whowouldwin.tsv",
    "xboxone.tsv",
    "yugioh.tsv"
]

# Get all TSV files from the ZIP
with zipfile.ZipFile("socialsent_subreddits.zip", "r") as z:
    archive_files = [
        file for file in z.namelist()
        if file.lower().endswith(".tsv")
    ]

print("TSV files found in ZIP:", len(archive_files))

# Match files by filename, ignoring folders and capitalization
archive_lookup = {
    os.path.basename(file).lower(): file
    for file in archive_files
}

# Load the requested subreddit lexicons
subreddit_lexicons = {}
missing_files = []

for filename in subreddit_files:
    actual_file = archive_lookup.get(filename.lower())

    if actual_file is None:
        missing_files.append(filename)
    else:
        subreddit_lexicons[filename] = load_lexicon_from_zip(
            "socialsent_subreddits.zip",
            actual_file
        )

print("Loaded subreddit lexicons:", len(subreddit_lexicons))
print("Missing files:", missing_files)

TSV files found in ZIP: 250
Loaded subreddit lexicons: 46
Missing files: ['occulus.tsv']


In [19]:
print("Historical adjective words:", len(adj_2000))
print("Historical frequent words:", len(freq_2000))
print("Subreddit lexicons:", len(subreddit_lexicons))

Historical adjective words: 2042
Historical frequent words: 4924
Subreddit lexicons: 46


In [20]:
# Combine all subreddit words into one set
all_subreddit_words = set()

for df in subreddit_lexicons.values():
    all_subreddit_words.update(df["word"].str.lower())

print("Unique subreddit words:", len(all_subreddit_words))

Unique subreddit words: 29963


In [21]:
adj_words = set(adj_2000["word"].str.lower())
freq_words = set(freq_2000["word"].str.lower())

print("Adjective words:", len(adj_words))
print("Frequent words:", len(freq_words))
print("Adjective/Frequent overlap:", len(adj_words & freq_words))
print("Adjective/Subreddit overlap:", len(adj_words & all_subreddit_words))
print("Frequent/Subreddit overlap:", len(freq_words & all_subreddit_words))

Adjective words: 2042
Frequent words: 4924
Adjective/Frequent overlap: 625
Adjective/Subreddit overlap: 1569
Frequent/Subreddit overlap: 4752


In [22]:
print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("Subreddit lexicons:", len(subreddit_lexicons))
print("Missing files:", missing_files)

Training: 45615
Validation: 2000
Test: 12284
Subreddit lexicons: 46
Missing files: ['occulus.tsv']


In [23]:
def prepare_lexicon(df):
    lexicon = df.copy()
    lexicon["word"] = lexicon["word"].astype(str).str.lower()
    return dict(zip(lexicon["word"], lexicon["score"]))

In [24]:
adj_lexicon = prepare_lexicon(adj_2000)
freq_lexicon = prepare_lexicon(freq_2000)

In [25]:
subreddit_combined = pd.concat(
    subreddit_lexicons.values(),
    ignore_index=True
)

subreddit_combined["word"] = (
    subreddit_combined["word"]
    .astype(str)
    .str.lower()
)

subreddit_combined = (
    subreddit_combined
    .groupby("word")["score"]
    .mean()
    .reset_index()
)

subreddit_lexicon = dict(
    zip(
        subreddit_combined["word"],
        subreddit_combined["score"]
    )
)

print("Historical adjective words:", len(adj_lexicon))
print("Historical frequent words:", len(freq_lexicon))
print("Combined subreddit words:", len(subreddit_lexicon))

Historical adjective words: 2042
Historical frequent words: 4924
Combined subreddit words: 29962


In [26]:
def extract_features(tweet):
    words = re.findall(r"\b\w+\b", tweet.lower())

    # ---------- Historical adjective features ----------
    adj_scores = [
        adj_lexicon[word]
        for word in words
        if word in adj_lexicon
    ]

    adj_positive = sum(score for score in adj_scores if score > 0)
    adj_negative = sum(score for score in adj_scores if score < 0)
    adj_count = len(adj_scores)

    # ---------- Historical frequent word features ----------
    freq_scores = [
        freq_lexicon[word]
        for word in words
        if word in freq_lexicon
    ]

    freq_positive = sum(score for score in freq_scores if score > 0)
    freq_negative = sum(score for score in freq_scores if score < 0)
    freq_count = len(freq_scores)

    # ---------- Subreddit features ----------
    subreddit_scores = [
        subreddit_lexicon[word]
        for word in words
        if word in subreddit_lexicon
    ]

    subreddit_positive = sum(
        score for score in subreddit_scores if score > 0
    )

    subreddit_negative = sum(
        score for score in subreddit_scores if score < 0
    )

    subreddit_count = len(subreddit_scores)

    # ---------- Text features ----------
    word_count = len(words)

    if words:
        longest_word = max(len(word) for word in words)
    else:
        longest_word = 0

    long_word_count = sum(
        1 for word in words if len(word) >= 5
    )

    # Use log(1 + x) so zero values do not cause problems
    feature_vector = [
        adj_positive,
        adj_negative,
        adj_count,
        freq_positive,
        freq_negative,
        freq_count,
        subreddit_positive,
        subreddit_negative,
        subreddit_count,
        math.log1p(word_count),
        math.log1p(longest_word),
        math.log1p(long_word_count)
    ]

    return feature_vector

In [27]:
features = extract_features(train_df.iloc[0]["tweet"])

print(features)
print("Number of features:", len(features))

[0.33, 0, 1, 0.74, -2.33, 5, 6.268279184583533, -0.3435294117647059, 12, 2.9444389791664403, 3.1780538303479458, 2.1972245773362196]
Number of features: 12


In [28]:
X_train = np.array([
    extract_features(tweet)
    for tweet in train_df["tweet"]
])

X_val = np.array([
    extract_features(tweet)
    for tweet in val_df["tweet"]
])

X_test = np.array([
    extract_features(tweet)
    for tweet in test_df["tweet"]
])

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)

X_train shape: (45615, 12)
X_val shape: (2000, 12)
X_test shape: (12284, 12)


In [29]:
y_train = train_df["label"].to_numpy()
y_val = val_df["label"].to_numpy()
y_test = test_df["label"].to_numpy()

print("y_train shape:", y_train.shape)
print("y_val shape:", y_val.shape)
print("y_test shape:", y_test.shape)

print("Classes:", np.unique(y_train))

y_train shape: (45615,)
y_val shape: (2000,)
y_test shape: (12284,)
Classes: [0 1 2]


In [30]:
X_train_bias = np.hstack([
    np.ones((X_train.shape[0], 1)),
    X_train
])

X_val_bias = np.hstack([
    np.ones((X_val.shape[0], 1)),
    X_val
])

X_test_bias = np.hstack([
    np.ones((X_test.shape[0], 1)),
    X_test
])

print("X_train_bias shape:", X_train_bias.shape)

X_train_bias shape: (45615, 13)


In [31]:
def softmax(z):
    z = z - np.max(z, axis=1, keepdims=True)
    exp_z = np.exp(z)
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

In [32]:
np.random.seed(42)

num_features = X_train_bias.shape[1]
num_classes = 3

W = np.zeros((num_features, num_classes))

print("Weight matrix shape:", W.shape)

Weight matrix shape: (13, 3)


In [33]:
def train_logistic_regression(X, y, W, learning_rate=0.01, epochs=100):
    n = X.shape[0]

    for epoch in range(epochs):

        # Forward pass
        scores = X @ W
        probabilities = softmax(scores)

        # One-hot encode labels
        y_one_hot = np.zeros_like(probabilities)
        y_one_hot[np.arange(n), y] = 1

        # Gradient
        gradient = (X.T @ (probabilities - y_one_hot)) / n

        # Update weights
        W -= learning_rate * gradient

        # Calculate loss
        loss = -np.mean(
            np.log(
                probabilities[np.arange(n), y] + 1e-15
            )
        )

        if (epoch + 1) % 10 == 0:
            print(
                f"Epoch {epoch + 1}/{epochs}, "
                f"Loss: {loss:.4f}"
            )

    return W

In [34]:
W = train_logistic_regression(
    X_train_bias,
    y_train,
    W,
    learning_rate=0.01,
    epochs=100
)

Epoch 10/100, Loss: 1.0117
Epoch 20/100, Loss: 0.9974
Epoch 30/100, Loss: 0.9865
Epoch 40/100, Loss: 0.9778
Epoch 50/100, Loss: 0.9709
Epoch 60/100, Loss: 0.9652
Epoch 70/100, Loss: 0.9605
Epoch 80/100, Loss: 0.9566
Epoch 90/100, Loss: 0.9532
Epoch 100/100, Loss: 0.9503


In [35]:
def predict_logistic_regression(X, W):
    scores = X @ W
    probabilities = softmax(scores)
    predictions = np.argmax(probabilities, axis=1)
    return predictions

In [36]:
y_pred = predict_logistic_regression(X_test_bias, W)

print("Predictions shape:", y_pred.shape)
print("First 20 predictions:", y_pred[:20])

Predictions shape: (12284,)
First 20 predictions: [1 1 1 1 1 1 1 2 1 1 2 1 2 1 1 1 1 0 1 1]


In [37]:
# Evaluate the model on the test dataset
test_accuracy = accuracy_score(y_test, y_pred)

weighted_f1 = f1_score(
    y_test,
    y_pred,
    average="weighted"
)

macro_f1 = f1_score(
    y_test,
    y_pred,
    average="macro"
)

print("Final Test Results")
print("------------------")
print(f"Accuracy: {test_accuracy:.4f}")
print(f"Weighted F1 Score: {weighted_f1:.4f}")
print(f"Macro F1 Score: {macro_f1:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        labels=[0, 1, 2],
        target_names=["Negative", "Neutral", "Positive"],
        zero_division=0
    )
)

Final Test Results
------------------
Accuracy: 0.4933
Weighted F1 Score: 0.4102
Macro F1 Score: 0.3790

Classification Report:


NameError: name 'classification_report' is not defined

In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        y_pred,
        target_names=["negative", "neutral", "positive"]
    )
)

In [ ]:
print("Training class distribution:")
print(train_df["label"].value_counts().sort_index())

print("\nTest class distribution:")
print(test_df["label"].value_counts().sort_index())

print("\nPredicted class distribution:")
print(pd.Series(y_pred).value_counts().sort_index())

In [ ]:
feature_names = [
    "adj_positive",
    "adj_negative",
    "adj_count",
    "freq_positive",
    "freq_negative",
    "freq_count",
    "subreddit_positive",
    "subreddit_negative",
    "subreddit_count",
    "log_word_count",
    "log_longest_word",
    "log_5plus_word_count"
]

for i, name in enumerate(feature_names):
    print(f"{i + 1}. {name}")

In [ ]:
feature_summary = pd.DataFrame(
    X_train,
    columns=feature_names
)

print(feature_summary.describe().T)

In [ ]:
print("\nNumber of unique values:")
print(feature_summary.nunique())

In [ ]:
results = pd.DataFrame({
    "Metric": [
        "Number of training tweets",
        "Number of validation tweets",
        "Number of test tweets",
        "Number of features",
        "Number of sentiment classes",
        "Test accuracy",
        "Weighted F1 score",
        "Macro F1 score"
    ],
    "Value": [
        len(y_train),
        len(y_val),
        len(y_test),
        X_train.shape[1],
        num_classes,
        round(test_accuracy, 4),
        round(weighted_f1, 4),
        round(macro_f1, 4)
    ]
})

display(results)